In [1]:
import sys, torch
sys.path.insert(0, ".")

from DNABERT2_modules import load_dnabert2
import transformers

print(f"transformers version: {transformers.__version__}")
tokenizer = transformers.AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", 
                                                       use_fast=False, trust_remote_code=True)


# Load pretrained weights into the local BertModel class.
# config_overrides lets you modify any BertConfig field before instantiation.
model, tokenizer = load_dnabert2(
    # config_overrides=None,   # e.g. {"num_hidden_layers": 6, "hidden_dropout_prob": 0.1}
    add_pooling_layer=False,
    config_overrides={"pad_token_id": tokenizer.pad_token_id}
)

device = next(model.parameters()).device
print(f"device     : {device}")
print(f"hidden_size: {model.config.hidden_size}")
print(f"num_layers : {model.config.num_hidden_layers}")

/home/andrew.dickson/.conda/envs/svar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers version: 4.42.4
device     : cuda:0
hidden_size: 768
num_layers : 12


In [2]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
inputs = tokenizer(dna, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

In [3]:
with torch.no_grad():
    hidden_states, _ = model(**inputs)  # (1, seq_len, 768)

# mean-pooled embedding (mask-weighted)
mask = inputs["attention_mask"].unsqueeze(-1).float()
embedding_mean = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1)
print(embedding_mean.shape)  # expect (1, 768)

# token-level hidden states
print(hidden_states.shape)  # expect (1, seq_len, 768)

torch.Size([1, 768])
torch.Size([1, 17, 768])


In [4]:
from transformers import AutoModel

# Only works for old versions of pytorch and transformers
ref_model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
ref_model = ref_model.to(device).eval()

# AutoModel uses the broken Triton path; patch it the same way our loader does.
from DNABERT2_modules.loader import _disable_triton_attention
_disable_triton_attention(ref_model)

with torch.no_grad():
    hidden_states, _ = model(**inputs)
    ref_hidden, _    = ref_model(**inputs)

max_output_diff = (hidden_states - ref_hidden).abs().max().item()
print(f"max output difference: {max_output_diff:.2e}")
assert max_output_diff == 0.0, f"outputs differ by {max_output_diff:.2e}"
print("outputs are bit-identical")

ValueError: The model class you are passing has a `config_class` attribute that is not consistent with the config class you passed (model has <class 'transformers.models.bert.configuration_bert.BertConfig'> and you passed <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>. Fix one of those so they match!